# 0.8 · 微积分 / Calculus

> **课程定位 / Where this fits**
> 第 8 课，**Part 0 · 基础准备**。
> Lesson 8, **Part 0 · Foundations**.
>
> 上一课线性代数解决"**怎么表达数据**"，这一课微积分解决"**怎么训练模型**"。**梯度下降、反向传播、损失最小化**——全是微积分。
> Linear algebra handles "how to represent data"; calculus handles "how to train models". **Gradient descent, backprop, loss minimization** all live here.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\frac{df}{dx}$, $f'(x)$ —— 一元导数 / scalar derivative
> - $\frac{\partial f}{\partial x_i}$ —— 偏导 / partial derivative
> - $\nabla f$ —— 梯度 / gradient（列向量）
> - $\nabla^2 f = \mathbf{H}$ —— Hessian
> - $\mathbf{J}$ —— Jacobian
> - $\eta$ —— 学习率 / learning rate
> - $\mathcal{O}(\cdot)$ —— 高阶项 / higher-order terms

> 💡 **面试相关 / Interview-relevant**
> - 让你手推 **logistic 回归的梯度**（出镜率 ★★★★★）
> - 解释 **反向传播 = 链式法则**（出镜率 ★★★★★）
> - 二阶条件判断 min/max/saddle（出镜率 ★★★）
>
> Whiteboard interview hits: derive logistic gradient, explain backprop = chain rule, second-order optimality.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释**导数 = 切线斜率 = 瞬时变化率 = 最佳线性近似**的等价性。
   Explain that derivative = tangent slope = instantaneous rate = best linear approximation.
2. 计算**多元函数的偏导、梯度、雅可比、海森**，并准确说明它们的形状。
   Compute partial derivatives, gradient, Jacobian, Hessian — and know their shapes.
3. 推导**链式法则**并解释为什么 backprop 本质就是它。
   Derive the chain rule and explain why backprop **is** the chain rule.
4. 用**梯度下降**最小化一个二维函数，并画出下降路径。
   Run gradient descent on a 2-D function and plot the trajectory.
5. **完全手写**一遍线性回归的梯度推导 + 梯度下降代码。
   Derive and implement linear-regression gradient descent **from scratch**.
6. 用 **PyTorch autograd** 验证手算的梯度，并理解 autodiff 的工作原理。
   Use **PyTorch autograd** to verify hand derivations and grasp how autodiff works.

---

## 目录 / Table of Contents

1. [一元导数 / Single-Variable Derivative](#1)
2. [常用导数表 / Derivative Cheat Sheet](#2)
3. [偏导数 / Partial Derivatives](#3)
4. [**梯度** ⭐ / Gradient](#4)
5. [方向导数 / Directional Derivative](#5)
6. [**链式法则** ⭐ / Chain Rule](#6)
7. [雅可比矩阵 / Jacobian](#7)
8. [Hessian 与二阶条件 / Hessian & Second-Order Conditions](#8)
9. [泰勒展开 / Taylor Expansion](#9)
10. [**梯度下降** ⭐ / Gradient Descent](#10)
11. [三种求导方式 / Three Ways to Compute Derivatives](#11)
12. [PyTorch Autograd 入门 / Intro to Autograd](#12)
13. [实战：手推 + 实现线性回归的梯度下降 / Hands-on](#13)
14. [小结 / Summary](#14)


<a id="1"></a>
## 1. 一元导数 / Single-Variable Derivative

### 1.1 定义 / Definition

$$
\boxed{\; f'(x) \;=\; \frac{df}{dx} \;=\; \lim_{h\to 0} \frac{f(x+h) - f(x)}{h} \;}
$$

三个等价的几何/物理解释 / Three equivalent views:
1. **切线斜率** / slope of the tangent at $x$
2. **瞬时变化率** / instantaneous rate of change
3. **最佳线性近似** / best linear approximation: $f(x+h) \approx f(x) + f'(x)\,h$

(3) 是最有用的一个——所有梯度下降的更新规则本质上都是 (3) 的逆用：**反着移动一个小步长 $h = -\eta f'(x)$ 来减小 $f$**。
(3) is the most useful — every gradient-descent step exploits it: move by $-\eta f'(x)$ to decrease $f$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# 演示：函数 f(x) = x³ - 3x，在 x=1.5 处的切线 / Tangent at x=1.5
def f(x): return x**3 - 3*x
def fprime(x): return 3*x**2 - 3        # 手算导数

x0 = 1.5
x = np.linspace(-2.5, 2.5, 200)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x, f(x), label="f(x) = x³ - 3x", linewidth=2)
ax.scatter([x0], [f(x0)], color="red", s=80, zorder=5)
# 切线：y = f(x0) + f'(x0)(x - x0)
tangent = f(x0) + fprime(x0) * (x - x0)
ax.plot(x, tangent, "r--", label=f"tangent at x={x0}, slope={fprime(x0):.2f}")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Derivative = slope of tangent line")
plt.show()


### 1.2 数值导数（中心差分）/ Numerical derivative via central difference

工程上你随时可以用差分近似导数：
You can always approximate the derivative by finite difference:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}, \quad h\text{ 取很小}$$

> **为什么用"中心差分"而不是 $\frac{f(x+h)-f(x)}{h}$？**
> 单边差分误差是 $\mathcal{O}(h)$，**中心差分是 $\mathcal{O}(h^2)$，精度高一个数量级**。
> Central diff has $\mathcal{O}(h^2)$ error vs forward diff's $\mathcal{O}(h)$ — one order tighter.


In [ ]:
def num_deriv(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print(f"analytic f'(1.5) = {fprime(1.5):.6f}")
print(f"numerical        = {num_deriv(f, 1.5):.6f}")
print(f"error            = {abs(fprime(1.5) - num_deriv(f, 1.5)):.2e}")


<a id="2"></a>
## 2. 常用导数表 / Derivative Cheat Sheet

DS 里反复出现的几个：
The handful you'll see again and again:

| $f(x)$ | $f'(x)$ |
|---|---|
| $x^n$ | $n x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| $\sin x$ | $\cos x$ |
| $\cos x$ | $-\sin x$ |
| $\sigma(x) = \dfrac{1}{1+e^{-x}}$ | $\sigma(x)\,(1-\sigma(x))$ ⭐ |
| $\tanh x$ | $1 - \tanh^2 x$ |
| $\max(0, x)$ (ReLU) | $\mathbb{1}\{x > 0\}$ |
| $\log\bigl(\sigma(x)\bigr)$ (log-sigmoid) | $1 - \sigma(x)$ ⭐ |

> ⭐ 标的两个 = 逻辑回归 / 神经网络反向传播里**每天都要用**。
> The starred ones come up daily in logistic regression / NN backprop.

### sigmoid 导数的妙处 / Why sigmoid's derivative is special

$$\sigma(x) = \frac{1}{1 + e^{-x}} \implies \sigma'(x) = \sigma(x)(1 - \sigma(x))$$

也就是：**算完前向 $\sigma(x)$ 后，导数无需再计算 $e^{-x}$，直接用 $\sigma\,(1-\sigma)$**。这就是为什么深度学习里 sigmoid 反向传播代码这么紧凑。
After computing $\sigma(x)$ forward, the backward pass costs **zero extra exponentials**.


In [ ]:
# 验证 sigmoid 导数公式 / Verify sigmoid derivative
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_prime_formula(x):
    s = sigmoid(x)
    return s * (1 - s)

x0 = 0.7
print(f"analytic σ'(0.7) = {sigmoid_prime_formula(x0):.6f}")
print(f"numerical        = {num_deriv(sigmoid, x0):.6f}")


<a id="3"></a>
## 3. 偏导数 / Partial Derivatives

多元函数 $f(x_1, x_2, \dots, x_n)$，**偏导**就是"把其他变量当常数"求导：
For multi-variable $f$, the partial derivative treats other variables as constants:

$$\frac{\partial f}{\partial x_i} = \lim_{h \to 0} \frac{f(x_1, \dots, x_i + h, \dots, x_n) - f(x_1, \dots, x_n)}{h}$$

记号也写作 $f_{x_i}$ 或 $\partial_i f$。
Also written $f_{x_i}$ or $\partial_i f$.

### 例 / Example

$$f(x, y) = x^2 + 3xy + y^3$$

$$\frac{\partial f}{\partial x} = 2x + 3y, \qquad \frac{\partial f}{\partial y} = 3x + 3y^2$$


In [ ]:
# 用 sympy 自动符号求偏导 / Symbolic partials via sympy
import sympy as sp

x, y = sp.symbols("x y", real=True)
f_sym = x**2 + 3*x*y + y**3

print("f(x,y) =", f_sym)
print("∂f/∂x  =", sp.diff(f_sym, x))
print("∂f/∂y  =", sp.diff(f_sym, y))


<a id="4"></a>
## 4. 梯度 ⭐ / Gradient

把所有偏导**装进一个列向量**就是梯度：
Stack all partials into a column vector = the gradient:

$$
\boxed{\;\nabla f(\mathbf{x}) \;=\; \begin{pmatrix} \dfrac{\partial f}{\partial x_1} \\\\[6pt] \dfrac{\partial f}{\partial x_2} \\\\ \vdots \\\\[6pt] \dfrac{\partial f}{\partial x_n} \end{pmatrix} \in \mathbb{R}^n\;}
$$

### 4.1 三个核心性质（一定要会）/ Three core properties

1. **方向**：$\nabla f(\mathbf{x})$ 指向 $f$ 在 $\mathbf{x}$ 处**上升最快**的方向。
   $\nabla f$ points in the direction of **steepest ascent**.
2. **大小**：$\|\nabla f(\mathbf{x})\|$ = 那个方向上的最大斜率。
   $\|\nabla f\|$ = the max slope.
3. **正交等高线**：$\nabla f(\mathbf{x}) \perp$ 经过 $\mathbf{x}$ 的等高线 / level set。
   $\nabla f$ is **perpendicular to the level set** through $\mathbf{x}$.

这三条加在一起 → **梯度下降**：朝 $-\nabla f$ 方向走，能最快减小 $f$。
Combine them → **gradient descent**: stepping along $-\nabla f$ decreases $f$ fastest.


In [ ]:
# 可视化：在等高线图上画梯度向量场 / Gradient field over contours
# f(x,y) = x² + 2y²  → 椭圆等高线 / elliptical level sets
def f2(x, y): return x**2 + 2*y**2
def grad_f2(x, y): return np.array([2*x, 4*y])

xs = np.linspace(-3, 3, 200); ys = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(xs, ys); Z = f2(X, Y)

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)

# 在网格上画梯度箭头 / Gradient arrows on a coarse grid
xs2 = np.linspace(-2.5, 2.5, 12); ys2 = np.linspace(-2.5, 2.5, 12)
Xq, Yq = np.meshgrid(xs2, ys2)
Gx, Gy = 2*Xq, 4*Yq
ax.quiver(Xq, Yq, Gx, Gy, color="red", alpha=0.7, width=0.003,
          scale=80, label="∇f (steepest ascent)")

ax.set_aspect("equal"); ax.legend()
ax.set_title("f(x,y) = x² + 2y²  with gradient field")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.show()


**观察 / Observation**:
- 梯度箭头**永远指向外侧**（远离原点 = $f$ 的最小值）—— 指向上升最快方向 ✓
- 椭圆扁平的方向（沿 $y$ 轴）箭头明显**比沿 $x$ 轴的更长**——因为 $f$ 沿 $y$ 上升更快（系数 2 vs 1）
- 每个箭头**垂直于经过该点的等高线** ✓

Gradient arrows always point **outward** (away from the minimum), are longer along $y$ where the function rises faster, and are perpendicular to level sets — all three properties confirmed.


<a id="5"></a>
## 5. 方向导数 / Directional Derivative

沿任意**单位向量** $\mathbf{u}$ 的方向，$f$ 的瞬时变化率：
The rate of change of $f$ along any unit vector $\mathbf{u}$:

$$D_{\mathbf{u}} f(\mathbf{x}) \;=\; \nabla f(\mathbf{x})^\top \mathbf{u} \;=\; \|\nabla f\|\,\cos\theta$$

其中 $\theta$ 是 $\nabla f$ 与 $\mathbf{u}$ 的夹角。
where $\theta$ is the angle between $\nabla f$ and $\mathbf{u}$.

由此立即知道：
- $\mathbf{u}$ 与 $\nabla f$ 同向 ($\theta = 0$) → 上升最快 ✓
- $\mathbf{u}$ 与 $\nabla f$ 反向 ($\theta = \pi$) → **下降最快**（这就是梯度下降）
- $\mathbf{u} \perp \nabla f$ → 沿等高线移动（值不变）

This is why $-\nabla f$ is the steepest **descent** direction — it's a direct corollary of $\cos\theta$ being minimized at $\theta = \pi$.


<a id="6"></a>
## 6. 链式法则 ⭐ / Chain Rule

### 6.1 一元链式法则 / Single-variable

如果 $y = f(u)$ 且 $u = g(x)$：
If $y = f(u)$ and $u = g(x)$:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = f'(u) \cdot g'(x)$$

### 6.2 多元链式法则 / Multivariate version

设 $z = f(\mathbf{u})$ 而 $\mathbf{u} = \mathbf{g}(\mathbf{x})$：
For $z = f(\mathbf{u})$ with $\mathbf{u} = \mathbf{g}(\mathbf{x})$:

$$\boxed{\;\frac{\partial z}{\partial x_i} = \sum_{j} \frac{\partial z}{\partial u_j}\,\frac{\partial u_j}{\partial x_i}\;}$$

矩阵形式 / Matrix form: $\nabla_{\mathbf{x}} z = \mathbf{J}_{\mathbf{g}}^\top \;\nabla_{\mathbf{u}} z$

### 6.3 为什么 backprop = 链式法则

神经网络是一长串嵌套函数：
A neural net is a chain of composed functions:

$$L = \ell\bigl(\,f^{(L)} \bigl(f^{(L-1)}(\dots f^{(1)}(\mathbf{x})\dots)\bigr),\; y\bigr)$$

对每层权重 $\mathbf{W}^{(k)}$ 求 $\partial L / \partial \mathbf{W}^{(k)}$ —— **链式法则从右往左一层层乘下去就行**。这就是反向传播。
For each layer's weight, the gradient is obtained by multiplying Jacobians **right to left**. That's literally backprop.

> 💡 **面试一句话答 / One-line interview answer**:
> "Backpropagation is the multivariate chain rule applied to a computational graph, executed in reverse topological order so we reuse intermediate Jacobians."


In [ ]:
# 演示：手算 + sympy 验证 链式法则 / Demo with sympy
# z = sin(x² + y²)
import sympy as sp
x, y = sp.symbols("x y", real=True)
z = sp.sin(x**2 + y**2)

# 直接求偏导 / Direct partial
print("∂z/∂x =", sp.diff(z, x))
# 手算：sin(u) where u = x² + y²
# ∂z/∂x = cos(u) · 2x = 2x·cos(x² + y²)


<a id="7"></a>
## 7. 雅可比矩阵 / Jacobian

当输出也是**向量**时，把所有偏导排成矩阵就是 Jacobian。
When the output is a vector too, stacking partials gives the Jacobian.

设 $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$，$\mathbf{f}(\mathbf{x}) = (f_1, \dots, f_m)^\top$：
For $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$:

$$
\mathbf{J}_{\mathbf{f}}(\mathbf{x}) = \frac{\partial \mathbf{f}}{\partial \mathbf{x}^\top}
= \begin{pmatrix}
\dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n} \\\\
\vdots & \ddots & \vdots \\\\
\dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n}
\end{pmatrix}
\in \mathbb{R}^{m \times n}
$$

### 形状记忆 / Shape mnemonic

- 输入 $n$ 维 → 列数 $n$
- 输出 $m$ 维 → 行数 $m$
- $\mathbf{J} \in \mathbb{R}^{m \times n}$

### 特殊情形 / Special cases

| $m, n$ | 简化为 / Reduces to |
|---|---|
| $m = 1, n = 1$ | 普通导数 $f'(x)$ |
| $m = 1, n > 1$ | **梯度** $\nabla f$（**行向量**形式）|
| $m > 1, n = 1$ | 向量对标量导数 |
| $m > 1, n > 1$ | 完整 Jacobian |

> ⚠ **梯度 vs Jacobian 的行/列 convention**：本仓库**梯度永远是列向量**（与 NOTATION.md 一致），所以 $\nabla f = \mathbf{J}_{f}^\top$（标量函数）。
> Per repo convention, gradients are **column vectors**; the Jacobian of a scalar function is a **row** vector, so $\nabla f = \mathbf{J}_f^\top$.


In [ ]:
# Jacobian 例子：极坐标 → 笛卡尔
# f(r, θ) = (r cos θ, r sin θ)
import sympy as sp
r, th = sp.symbols("r theta", real=True)

f_vec = sp.Matrix([r * sp.cos(th), r * sp.sin(th)])
J = f_vec.jacobian([r, th])
print("Jacobian J =")
sp.pprint(J)
print(f"\ndet(J) =", sp.simplify(J.det()))
# det = r —— 这就是为什么极坐标的面积元是 r dr dθ


<a id="8"></a>
## 8. Hessian 与二阶条件 / Hessian & Second-Order Conditions

**所有二阶偏导**装成方阵：
Stack **all second-order** partials into a square matrix:

$$
\mathbf{H}(\mathbf{x}) = \nabla^2 f(\mathbf{x}) = \begin{pmatrix}
\dfrac{\partial^2 f}{\partial x_1^2} & \cdots & \dfrac{\partial^2 f}{\partial x_1 \partial x_n} \\\\
\vdots & \ddots & \vdots \\\\
\dfrac{\partial^2 f}{\partial x_n \partial x_1} & \cdots & \dfrac{\partial^2 f}{\partial x_n^2}
\end{pmatrix}
\in \mathbb{R}^{n \times n}
$$

**Schwarz 定理**：$f$ 二阶连续可微 ⇒ Hessian 对称（$\partial_i \partial_j f = \partial_j \partial_i f$）。
**Schwarz**: smooth $f$ ⇒ Hessian is symmetric.

### 临界点的分类 / Classifying critical points

在 $\nabla f(\mathbf{x}^*) = \mathbf{0}$ 处：
At critical points where $\nabla f = \mathbf{0}$:

| $\mathbf{H}$ | 意义 / Meaning |
|---|---|
| 正定 / positive definite (所有 $\lambda_i > 0$) | **局部极小** / local min |
| 负定 / negative definite (所有 $\lambda_i < 0$) | 局部极大 / local max |
| **不定** / indefinite（有正有负 $\lambda$）| **鞍点** ⭐ / saddle |
| 半定 / semi-definite (有 $\lambda = 0$) | 需要更高阶判断 / inconclusive |

> ⚠ **深度学习的"恐怖"事实 / DL gotcha**:
> 高维空间里**临界点几乎都是鞍点**，不是局部最小。这就是为什么 SGD 在高维 NN 训练里跑得动——它能逃出鞍点。
> In high dimensions, **most critical points are saddles**, not local minima. SGD's noise helps escape them — that's a key reason it works in deep learning.


In [ ]:
# 鞍点可视化 / Saddle point viz
# f(x,y) = x² - y²，在 (0,0) 是鞍点 / saddle at origin
xs = np.linspace(-2, 2, 100); ys = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(xs, ys)
Z = X**2 - Y**2

fig = plt.figure(figsize=(12, 5))

# 左：3D 表面 / 3D surface
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(X, Y, Z, cmap="coolwarm", alpha=0.85, edgecolor="none")
ax.scatter([0], [0], [0], color="black", s=80)
ax.set_title("f(x,y) = x² - y²  (saddle at origin)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("f")

# 右：等高线 + 梯度 / Right: contour + gradient
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(X, Y, Z, levels=20, cmap="coolwarm")
ax2.scatter([0], [0], color="black", s=80, zorder=5, label="saddle ∇f=0")
ax2.set_aspect("equal"); ax2.legend()
ax2.set_title("Hessian = [[2,0],[0,-2]]  →  λ = {+2, -2}  →  saddle")
plt.tight_layout(); plt.show()


In [ ]:
# 用 sympy 求 Hessian
x, y = sp.symbols("x y", real=True)
f_expr = x**2 - y**2

grad = sp.Matrix([sp.diff(f_expr, x), sp.diff(f_expr, y)])
H = sp.hessian(f_expr, (x, y))
print("∇f =", grad.T)
print("\nHessian H =")
sp.pprint(H)
print(f"\neigenvalues: {sorted(H.eigenvals().keys())}")
# 特征值 {+2, -2} 一正一负 → indefinite → saddle


<a id="9"></a>
## 9. 泰勒展开 / Taylor Expansion

任何"够光滑"的函数在 $\mathbf{x}_0$ 附近都可以用多项式近似：
Smooth functions admit polynomial approximations near $\mathbf{x}_0$:

### 一元 / Single-variable

$$f(x_0 + h) = f(x_0) + f'(x_0)\,h + \tfrac{1}{2}f''(x_0)\,h^2 + \mathcal{O}(h^3)$$

### 多元 / Multivariate

$$f(\mathbf{x}_0 + \mathbf{h}) = f(\mathbf{x}_0) + \nabla f(\mathbf{x}_0)^\top \mathbf{h} + \tfrac{1}{2}\mathbf{h}^\top \mathbf{H}(\mathbf{x}_0)\,\mathbf{h} + \mathcal{O}(\|\mathbf{h}\|^3)$$

### 用途 / Uses

- **梯度下降**用一阶项 / GD uses the **first-order** term
- **Newton 法**用一阶 + 二阶项 / Newton uses **first + second**
- 训练 LLM 时的 **AdamW** 也是基于一阶 + 历史信息 / Adam uses 1st-order + history


In [ ]:
# 演示 Taylor 近似如何随阶数收紧 / Taylor approximations of different orders
from math import factorial      # NumPy 2.x 移除了 np.math / np.math removed in NumPy 2.x

def true_f(x): return np.exp(x)
def taylor(x, order):
    out = np.zeros_like(x)
    for k in range(order + 1):
        out += x**k / factorial(k)
    return out

x = np.linspace(-2, 2, 200)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, true_f(x), label="exp(x)", color="black", linewidth=2)
for order in [1, 2, 4, 8]:
    ax.plot(x, taylor(x, order), "--", label=f"Taylor order {order}")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Taylor approximations of exp(x) around x=0")
plt.show()


<a id="10"></a>
## 10. 梯度下降 ⭐ / Gradient Descent

### 10.1 算法 / The algorithm

要最小化 $f(\mathbf{x})$，**反梯度方向迭代**：
To minimize $f$, iterate **opposite the gradient**:

$$
\boxed{\;\mathbf{x}^{(t+1)} \;=\; \mathbf{x}^{(t)} \;-\; \eta\,\nabla f(\mathbf{x}^{(t)})\;}
$$

其中 $\eta > 0$ 是**学习率** / learning rate。

### 10.2 为什么这样能下降 / Why does it decrease $f$

一阶 Taylor：
$$f(\mathbf{x} - \eta\nabla f) \approx f(\mathbf{x}) - \eta\,\|\nabla f\|^2 < f(\mathbf{x})$$

只要 $\eta$ 足够小 + $\nabla f \ne \mathbf{0}$，每步严格下降。**$\eta$ 太大就可能"步子太大跨过谷底"**——所以调 $\eta$ 是 DL 的核心痛点。
As long as $\eta$ is small enough and the gradient is nonzero, each step strictly decreases $f$. **Too large $\eta$ overshoots** — tuning $\eta$ is a central DL pain point.

### 10.3 学习率太小、太大、刚好 / Three regimes


In [ ]:
# 演示：在 f(x,y) = x² + 2y² 上跑梯度下降，比较 3 个学习率
# Run GD on f(x,y) = x² + 2y² with three learning rates
def grad_f2(x): return np.array([2*x[0], 4*x[1]])

def gd(x0, eta, n_steps=30):
    xs = [x0]
    for _ in range(n_steps):
        xs.append(xs[-1] - eta * grad_f2(xs[-1]))
    return np.array(xs)

xs_grid = np.linspace(-3, 3, 200); ys_grid = np.linspace(-3, 3, 200)
Xg, Yg = np.meshgrid(xs_grid, ys_grid); Zg = Xg**2 + 2*Yg**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (eta, label) in zip(axes,
        [(0.05, "η=0.05 (too small)"), (0.25, "η=0.25 (just right)"), (0.55, "η=0.55 (overshoot)")]):
    ax.contour(Xg, Yg, Zg, levels=15, cmap="viridis", alpha=0.6)
    traj = gd(np.array([2.7, 2.5]), eta)
    ax.plot(traj[:, 0], traj[:, 1], "o-", color="red", markersize=4, linewidth=1.2)
    ax.scatter(traj[0, 0], traj[0, 1], color="blue", s=80, zorder=5, label="start")
    ax.scatter(0, 0, color="green", marker="*", s=200, zorder=5, label="minimum")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=9)
    ax.set_title(label)
plt.tight_layout(); plt.show()


**三种学习率的行为 / Behavior across regimes**:

- **太小 (η=0.05)**：方向对，但走得太慢，30 步后还没到底
- **刚好 (η=0.25)**：几步就稳稳收敛到最小值
- **太大 (η=0.55)**：**沿陡峭方向震荡**——在 $y$ 方向（二阶导更大）反复跨过谷底

这正是**预条件 / 自适应学习率**（Adam、RMSProp）要解决的问题：每个方向用不同的学习率。
This motivates **adaptive learning rates** (Adam, RMSProp) — different step size per direction.


<a id="11"></a>
## 11. 三种求导方式 / Three Ways to Compute Derivatives

| 方式 / Way | 工具 / Tool | 优 / Pros | 缺 / Cons |
|---|---|---|---|
| 符号 / Symbolic | sympy | 精确公式 | 表达式会爆炸（DL 不可用）|
| 数值 / Numerical | finite difference | 通用、简单 | 精度差、慢、**梯度噪声大** |
| **自动微分** / Autodiff | PyTorch / JAX / TF | 精度=符号，速度~原函数 | 需要把代码"穿"进框架 |

> **现代深度学习 = 自动微分 + GPU 数值线代**。
> Modern DL = autodiff + GPU linear algebra.


In [ ]:
# 三种方式对比：f(x) = x³ - 3x，求 x=2 处导数
# Compare: f(x) = x³ - 3x at x=2
import sympy as sp
import torch

x_sym = sp.symbols("x")
f_sym = x_sym**3 - 3 * x_sym
analytic = sp.diff(f_sym, x_sym).subs(x_sym, 2)
print(f"symbolic : {float(analytic)}")

def f_np(x): return x**3 - 3*x
numerical = (f_np(2 + 1e-5) - f_np(2 - 1e-5)) / (2 * 1e-5)
print(f"numerical: {numerical}")

x_t = torch.tensor(2.0, requires_grad=True)
y_t = x_t**3 - 3 * x_t
y_t.backward()
print(f"autograd : {x_t.grad.item()}")


<a id="12"></a>
## 12. PyTorch Autograd 入门 / Intro to Autograd

PyTorch 的 `autograd` 是**所有深度学习的引擎**。原理：在你写 `y = f(x)` 的时候，它**默默搭一棵计算图**；当你调用 `y.backward()`，它**反向遍历**这棵图并应用链式法则。
PyTorch's `autograd` is the engine behind all deep learning. It builds a computation graph as you write `y = f(x)`, then `.backward()` walks it **in reverse** applying the chain rule.

### 三步流程 / Three-step flow
1. 标记需要求导的张量 `requires_grad=True`
2. 算出最终的标量损失 `loss`
3. 调 `loss.backward()` —— 梯度自动写到 `.grad` 上


In [ ]:
# 简单例子 / Simple example
import torch
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

z = x**2 * y + y**3       # z 是一个标量函数 f(x,y) = x²y + y³
z.backward()              # 一次反向，把梯度填进 x.grad, y.grad

# 手算：∂z/∂x = 2xy = 24    ∂z/∂y = x² + 3y² = 57
print(f"∂z/∂x = {x.grad.item()}  (expect 24)")
print(f"∂z/∂y = {y.grad.item()}  (expect 57)")


In [ ]:
# 向量版：算梯度 / Vector input
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (w**2).sum()              # f(w) = w₁² + w₂² + w₃²
loss.backward()
print(f"∇f(w) = {w.grad.tolist()}")
# Expected: 2w = [2, 4, 6]


### 🧠 Autograd 的工作原理简述 / How autograd works

```
            forward                              backward
   x ───→ x² ───→ * ───→ sum ──→ loss     |     loss ──→ sum ──→ * ──→ x² ──→ x
                              ↑           |       │       │      │      │
                       requires_grad      |       1.0   ∂/∂*   ∂/∂x²   x.grad
```

- **正向**：每个运算把输入张量记入计算图节点
- **反向**：从 `loss` 开始，每个节点根据**自己定义的 backward 函数**把上游梯度传给输入

每个 PyTorch 算子（`+ * @ sin exp` ...）都自带 backward 函数，所以你**写正常的 Python 代码就能拿到任意梯度**——这就是 DL 框架的魔法。

Every PyTorch op ships with a custom backward function, so writing normal Python forward code automatically grants you gradients of arbitrary depth.


<a id="13"></a>
## 13. 实战：手推 + 实现线性回归的梯度下降 / Hands-on

完整走一遍：手算梯度 → NumPy GD → 和 autograd 对比 → 检查收敛。
End-to-end: derive gradient by hand → NumPy GD → cross-check with autograd → check convergence.

### 数学推导 / Derivation

数据 $\mathbf{X} \in \mathbb{R}^{n\times d}$，标签 $\mathbf{y} \in \mathbb{R}^n$，参数 $\mathbf{w} \in \mathbb{R}^d$（先不要 bias）。

损失（MSE）：
$$J(\mathbf{w}) = \frac{1}{n} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 = \frac{1}{n}(\mathbf{y} - \mathbf{X}\mathbf{w})^\top (\mathbf{y} - \mathbf{X}\mathbf{w})$$

展开：
$$J = \frac{1}{n}(\mathbf{y}^\top \mathbf{y} - 2\mathbf{w}^\top \mathbf{X}^\top \mathbf{y} + \mathbf{w}^\top \mathbf{X}^\top \mathbf{X} \mathbf{w})$$

对 $\mathbf{w}$ 求梯度（用矩阵微积分恒等式 $\nabla_{\mathbf{w}}(\mathbf{w}^\top \mathbf{A} \mathbf{w}) = (\mathbf{A} + \mathbf{A}^\top)\mathbf{w}$；这里 $\mathbf{A} = \mathbf{X}^\top\mathbf{X}$ 对称）：

$$\boxed{\;\nabla J(\mathbf{w}) = \frac{2}{n} \mathbf{X}^\top (\mathbf{X}\mathbf{w} - \mathbf{y})\;}$$

> ⭐ **必须背下来**——线性回归 / 神经网络最后一层一直用。
> **Memorize this** — used in linear regression and the last layer of every NN.

更新规则 / Update rule:
$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \cdot \frac{2}{n}\mathbf{X}^\top(\mathbf{X}\mathbf{w}^{(t)} - \mathbf{y})$$


In [ ]:
# 造数据 / Synthesize data
rng = np.random.default_rng(0)
n, d = 200, 3
X = rng.normal(size=(n, d))
true_w = np.array([1.5, -2.0, 0.5])
y = X @ true_w + 0.3 * rng.normal(size=n)         # 加噪声 / add noise

# 我们手算的梯度函数 / Manual gradient
def J(w):     return ((y - X @ w)**2).mean()
def gradJ(w): return (2 / n) * X.T @ (X @ w - y)


In [ ]:
# 用 GD 跑一遍 / Run GD
w = np.zeros(d)
eta = 0.1
losses = []
for t in range(50):
    losses.append(J(w))
    w = w - eta * gradJ(w)

print(f"true w  : {true_w}")
print(f"learnt w: {w.round(3)}")
print(f"final loss: {losses[-1]:.5f}")


In [ ]:
# 损失收敛曲线 / Loss curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses, "o-", markersize=4)
ax.set_xlabel("iteration"); ax.set_ylabel("MSE loss")
ax.set_yscale("log")
ax.set_title("Gradient descent on linear regression — MSE converges")
ax.grid(alpha=0.3)
plt.show()


In [ ]:
# 用 autograd 验证我们手推的梯度公式正确 / Verify the derivation via autograd
import torch
Xt = torch.tensor(X, dtype=torch.float64)
yt = torch.tensor(y, dtype=torch.float64)
w_t = torch.tensor(rng.normal(size=d), dtype=torch.float64, requires_grad=True)

loss = ((yt - Xt @ w_t)**2).mean()
loss.backward()

manual = (2 / n) * X.T @ (X @ w_t.detach().numpy() - y)

print(f"autograd ∇J : {w_t.grad.numpy().round(6)}")
print(f"manual   ∇J : {manual.round(6)}")
print(f"max diff    : {np.abs(w_t.grad.numpy() - manual).max():.2e}")


**手推梯度与 autograd 完全一致**（误差 ~ $10^{-14}$，浮点精度）—— 公式背对了。
**Hand derivation matches autograd to float precision** — formula correct.

📝 这就把本节核心串了起来：
This ties together:
- 偏导 → 梯度（向量）
- 矩阵微积分恒等式（$\nabla \mathbf{w}^\top \mathbf{A} \mathbf{w}$）
- 链式法则隐含在 `loss.backward()` 里
- 梯度下降算法 + 收敛监控


<a id="14"></a>
## 14. 小结 / Summary

### 概念地图 / Concept map

```
导数 (slope)
   │
   ├── 偏导
   │     ├── 梯度 ∇f ── 步骤"反向走" → 梯度下降
   │     │      │
   │     │      └── 方向导数 = ∇f · u
   │     │
   │     └── 雅可比 J (输出也是向量时)
   │              │
   │              └── 链式法则的"矩阵版"
   │                          │
   │                          └── backprop ⭐
   │
   └── 二阶导
         ├── Hessian H = ∇²f
         │      ├── 对称（Schwarz）
         │      ├── 正定 → min；负定 → max；不定 → saddle ⭐
         │      └── Newton 法、Adam、L-BFGS
         │
         └── Taylor 二阶展开
                │
                └── 解释优化算法收敛
```

### 🧠 必背公式 / Must-know formulas

| 表达 | 梯度 |
|---|---|
| $\mathbf{a}^\top \mathbf{x}$ | $\mathbf{a}$ |
| $\mathbf{x}^\top \mathbf{A} \mathbf{x}$ | $(\mathbf{A} + \mathbf{A}^\top)\mathbf{x}$（$\mathbf{A}$ 对称时 $= 2\mathbf{A}\mathbf{x}$）|
| $\|\mathbf{x}\|_2^2$ | $2\mathbf{x}$ |
| $\|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2$ | $-2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$ |
| $\sigma(z) = (1+e^{-z})^{-1}$ | $\sigma(z)(1-\sigma(z))$ |
| 交叉熵 $-\log \sigma(z)$（label=1）| $\sigma(z) - 1$ |

### 💡 工业速查 / Industry cheat sheet

```python
# 手推梯度  ← 面试 + 论文阅读
∇J(w) = (2/n) * X.T @ (X @ w - y)        # 线性回归 / linear regression
∇J(w) = (1/n) * X.T @ (sigmoid(X@w) - y) # 逻辑回归 / logistic regression

# 数值梯度（检查公式有没有写错）
(f(w + h*e) - f(w - h*e)) / (2*h)

# PyTorch autograd 三步
w = torch.tensor(..., requires_grad=True)
loss = ...                               # 必须是标量 / must be scalar
loss.backward()                          # w.grad 拿到梯度

# 梯度下降一行
w -= eta * w.grad
w.grad.zero_()                           # 别忘了清零！/ don't forget zero_grad
```

### 💡 面试速查 / Interview must-knows

1. **梯度的三条性质**：方向最陡 / 大小最大 / 垂直等高线
2. **链式法则 = backprop**：高维 = Jacobian 乘法
3. **临界点分类**：用 Hessian 特征值符号
4. **高维"鞍点泛滥"** → SGD 噪声有助逃逸
5. **手推 LinReg / LogReg 梯度**：公式必背
6. **学习率 η** 太大震荡、太小慢、Adam 自适应

### 下一节预告 / Next up

**Part 0.9 · 概率论** —— 随机变量、期望、方差、各种分布、贝叶斯公式。**机器学习里所有 "推断/不确定性" 的语言**都建立在概率上。
**Part 0.9 · Probability** — random variables, expectation, variance, distributions, Bayes. **The language of all inference and uncertainty in ML.**
